# Chapter 8 — Heteroscedasticity
## University of Neuchâtel — Python for Econometrics

Topics:
1. What is heteroscedasticity and why it matters
2. Visual detection: residual plots
3. Breusch-Pagan test (manual + statsmodels)
4. White test (manual + statsmodels)
5. Heteroscedasticity-robust standard errors (HC1)
6. Weighted Least Squares (WLS)

**Format:** One cell per step. Manual first, statsmodels second, `np.isclose()` comparison.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
import statsmodels.stats.diagnostic as diag
from scipy.stats import chi2
import matplotlib.pyplot as plt
import wooldridge as woo
import os

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

## 1 — What is Heteroscedasticity?

**Homoscedasticity (OLS assumption):** $Var(u|x) = \sigma^2$ (constant variance)

**Heteroscedasticity:** $Var(u|x_i) = \sigma_i^2$ (variance varies with $x$)

**Consequences:**
- OLS is still **unbiased** and **consistent** ✓
- But OLS standard errors are **biased** → t and F tests are invalid ✗
- OLS is no longer BLUE (Best Linear Unbiased Estimator)

**When does it arise?** Very common in cross-sectional data (wages, house prices, firm sizes).

## 2 — Load Data: `hprice1`

In [ ]:
df = woo.dataWoo('hprice1')
print('Shape:', df.shape)
print(df[['price','lotsize','sqrft','bdrms']].head(10))

In [ ]:
print('Descriptive statistics:')
print(df[['price','lotsize','sqrft','bdrms']].describe().round(2))

## 3 — Baseline OLS: price ~ lotsize + sqrft + bdrms

### Step 3.1 — Manual OLS

In [ ]:
n = len(df)
X = np.column_stack([
    np.ones(n),
    df['lotsize'].values,
    df['sqrft'].values,
    df['bdrms'].values
])
y = df['price'].values
beta_hat = np.linalg.solve(X.T @ X, X.T @ y)
labels = ['const','lotsize','sqrft','bdrms']
print('Manual OLS:')
for lbl, b in zip(labels, beta_hat):
    print(f'  {lbl:10s}: {b:.4f}')

### Step 3.2 — Statsmodels OLS

In [ ]:
res = smf.ols('price ~ lotsize + sqrft + bdrms', data=df).fit()
print(res.summary())

### Step 3.3 — Compare

In [ ]:
sm_p = res.params
print('Coefficient comparison:')
for lbl, b_man in zip(labels, beta_hat):
    match = np.isclose(b_man, sm_p[lbl])
    print(f'  {lbl:10s}: manual={b_man:.4f}, sm={sm_p[lbl]:.4f}, match={match}')

## 4 — Visual Detection of Heteroscedasticity

In [ ]:
e = res.resid.values
y_hat = res.fittedvalues.values
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].scatter(y_hat, e, alpha=0.5, s=20)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')
axes[1].scatter(y_hat, np.abs(e), alpha=0.5, s=20)
axes[1].set_xlabel('Fitted Values')
axes[1].set_ylabel('|Residuals|')
axes[1].set_title('|Residuals| vs Fitted')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch08_01_residuals.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Fanning pattern → evidence of heteroscedasticity')

## 5 — Breusch-Pagan Test

**$H_0$:** $Var(u|x) = \sigma^2$ (homoscedasticity)

**Manual procedure:**
1. Get OLS residuals, compute $\hat{u}^2$
2. Regress $\hat{u}^2$ on regressors
3. $LM = n \cdot R^2_{auxiliary}$
4. Under $H_0$: $LM \sim \chi^2(k)$

### Step 5.1 — Compute u² and auxiliary regression

In [ ]:
u2 = e**2
df['u2'] = u2
print('Squared residuals added (first 5 rows):')
print(df[['price','u2']].head())

In [ ]:
res_bp_aux = smf.ols('u2 ~ lotsize + sqrft + bdrms', data=df).fit()
R2_aux = res_bp_aux.rsquared
LM_manual = n * R2_aux
pval_manual = chi2.sf(LM_manual, df=3)
print(f'BP: R2_aux={R2_aux:.4f}, LM={LM_manual:.4f}, p={pval_manual:.4f}')
if pval_manual < 0.05:
    print('Reject H0: heteroscedasticity detected')
else:
    print('Fail to reject H0: no strong evidence of heteroscedasticity')

### Step 5.2 — Breusch-Pagan with statsmodels

In [ ]:
bp_test = diag.het_breuschpagan(e, X[:, 1:])
print('BP test (statsmodels):')
print(f'  LM statistic: {bp_test[0]:.4f}')
print(f'  p-value:      {bp_test[1]:.4f}')
print(f'  LM match: {np.isclose(LM_manual, bp_test[0], rtol=0.01)}')

## 6 — White Test

More general than BP — includes squares and cross-products of regressors.

This detects more forms of heteroscedasticity.

### Step 6.1 — Create squares and cross-products manually

In [ ]:
df['lotsize2'] = df['lotsize']**2
df['sqrft2']   = df['sqrft']**2
df['bdrms2']   = df['bdrms']**2
df['lot_sq']   = df['lotsize'] * df['sqrft']
df['lot_bd']   = df['lotsize'] * df['bdrms']
df['sq_bd']    = df['sqrft']   * df['bdrms']
print('Added squares and cross-products (first 3 rows):')
print(df[['lotsize2','sqrft2','bdrms2','lot_sq','lot_bd','sq_bd']].head(3))

In [ ]:
res_w_aux = smf.ols(
    'u2 ~ lotsize + sqrft + bdrms + lotsize2 + sqrft2 + bdrms2 + lot_sq + lot_bd + sq_bd',
    data=df
).fit()
LM_white = n * res_w_aux.rsquared
pval_white = chi2.sf(LM_white, df=9)
print(f'White test (manual): LM={LM_white:.4f}, p={pval_white:.4f}')

### Step 6.2 — White test with statsmodels

In [ ]:
wt = diag.het_white(e, X)
print(f'White test (statsmodels): LM={wt[0]:.4f}, p={wt[1]:.4f}')
print(f'Match: {np.isclose(LM_white, wt[0], rtol=0.01)}')

## 7 — Robust Standard Errors (HC1)

Instead of GLS/WLS, we can use **heteroscedasticity-robust SEs** (sandwich estimator):

$$\widehat{Var}_{HC1}(\hat{\beta}) = \frac{n}{n-k-1}(X'X)^{-1}\left(\sum_i \hat{u}_i^2 x_i x_i'\right)(X'X)^{-1}$$

OLS coefficients stay the same — only the SEs change.

### Step 7.1 — Robust SEs with statsmodels HC1

In [ ]:
res_hc1 = res.get_robustcov_results(cov_type='HC1')
print('Robust (HC1) standard errors:')
print(res_hc1.summary())

### Step 7.2 — Compare OLS vs Robust SEs

In [ ]:
print('Standard error comparison (OLS vs HC1):')
for lbl in labels:
    se_ols = res.bse[lbl]
    se_hc1 = res_hc1.bse[lbl]
    print(f'  {lbl:10s}: OLS SE={se_ols:.4f},  HC1 SE={se_hc1:.4f},  ratio={se_hc1/se_ols:.3f}')

## 8 — Weighted Least Squares (WLS)

If $Var(u_i|x_i) = \sigma^2 h_i$ for known $h_i$, WLS is BLUE:

Transform: multiply all variables by $1/\sqrt{h_i}$ and apply OLS.

**Here:** assume $Var(u|x) \propto lotsize$, so weights $w_i = 1/lotsize_i$.

### Step 8.1 — Manual WLS via transformation

In [ ]:
w = 1.0 / df['lotsize'].values
sqrt_w = np.sqrt(w)
X_wls = (X.T * sqrt_w).T
y_wls = y * sqrt_w
beta_wls = np.linalg.solve(X_wls.T @ X_wls, X_wls.T @ y_wls)
print('WLS coefficients (manual):')
for lbl, b in zip(labels, beta_wls):
    print(f'  {lbl:10s}: {b:.4f}')

### Step 8.2 — WLS with statsmodels

In [ ]:
res_wls = smf.wls('price ~ lotsize + sqrft + bdrms', data=df, weights=1/df['lotsize']).fit()
print(res_wls.summary())

### Step 8.3 — Compare WLS manual vs statsmodels

In [ ]:
sm_wls = res_wls.params
print('WLS comparison:')
for lbl, b_man in zip(labels, beta_wls):
    match = np.isclose(b_man, sm_wls[lbl])
    print(f'  {lbl:10s}: manual={b_man:.4f}, sm={sm_wls[lbl]:.4f}, match={match}')

## 9 — Summary Table: OLS vs Robust vs WLS

In [ ]:
sumtab = pd.DataFrame({
    'Variable': labels,
    'OLS_b':    [res.params[l] for l in labels],
    'OLS_SE':   [res.bse[l] for l in labels],
    'HC1_SE':   [res_hc1.bse[l] for l in labels],
    'WLS_b':    beta_wls,
    'WLS_SE':   [res_wls.bse[l] for l in labels]
})
print(sumtab.round(4).to_string(index=False))
sumtab.to_csv(os.path.join(output_dir,'table_ch08_hetero_comparison.csv'), index=False)
print('Saved table_ch08_hetero_comparison.csv')